# 71 — Train BGE-reranker-v2-m3 cross-encoder (Stage B)

Re-mines HNs via the Stage A fine-tuned BGE-M3, then full-FT the
cross-encoder via sentence-transformers' CrossEncoder API.

**Prereqs**: Stage A complete — `OrRim123/recsys2026-bge-m3-music-v1-merged`
exists on Hub. HF_TOKEN in Colab Secrets.

**Wallclock**: ~3-5 hr on Blackwell (1 hr HN re-mine + 2-4 hr train).

In [ ]:
# 1) Setup — clone + HF auth + Drive mount + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
os.makedirs(src, exist_ok=True)
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

!pip install -q --upgrade \
    'transformers>=4.40' 'accelerate>=0.30' \
    'sentence-transformers>=3.0' \
    'datasets' 'pandas<3.0' 'tqdm'

In [ ]:
# 2) Smoke HN re-mine: 200 rows, verify the script runs.
!cd /content/recsys2026 && python scripts/build_cross_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --bge-m3-ft-hub OrRim123/recsys2026-bge-m3-music-v1-merged \
    --output experiments/cache/retrieval_v2/triples_reranker_smoke.jsonl \
    --max-rows 200 --k-negs 7 \
    2>&1 | tail -10
!wc -l experiments/cache/retrieval_v2/triples_reranker_smoke.jsonl

In [ ]:
# 3) Full HN re-mine — ~1 hr on Blackwell.
!cd /content/recsys2026 && python -u scripts/build_cross_encoder_training_data.py \
    --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
    --bge-m3-ft-hub OrRim123/recsys2026-bge-m3-music-v1-merged \
    --output experiments/cache/retrieval_v2/triples_reranker.jsonl \
    --percpos-threshold 0.80 --k-negs 7 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_hn_remine_log.txt
!wc -l experiments/cache/retrieval_v2/triples_reranker.jsonl

In [ ]:
# 4) Smoke fine-tune: 1 epoch on 500 triples.
!head -500 experiments/cache/retrieval_v2/triples_reranker.jsonl > experiments/cache/retrieval_v2/triples_reranker_smoke_500.jsonl
!cd /content/recsys2026 && python scripts/train_cross_encoder.py \
    --triples experiments/cache/retrieval_v2/triples_reranker_smoke_500.jsonl \
    --output-dir /content/bge_reranker_smoke \
    --hub-repo OrRim123/recsys2026-bge-reranker-smoke \
    --epochs 1 --batch-size 8 \
    2>&1 | tail -15
!rm -rf /content/bge_reranker_smoke

In [ ]:
# 5) FULL fine-tune: ~2-4 hr on Blackwell. Pushes to Hub.
!cd /content/recsys2026 && python -u scripts/train_cross_encoder.py \
    --triples experiments/cache/retrieval_v2/triples_reranker.jsonl \
    --output-dir /content/bge_reranker_finetune \
    --hub-repo OrRim123/recsys2026-bge-reranker-music-v1 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/ce_train_log.txt

In [ ]:
# 6) Offline eval (Stage A+B): nDCG@20 on dev split through full pipeline.
# Pulls dev conversations from HF, computes BM25+dense_lyrics+BGE-M3-FT top-100,
# then reranks with the fine-tuned BGE-reranker. Gate: nDCG@20 >= 0.25.
import sys, math, os, pickle
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sys.path.insert(0, '/content/recsys2026/scripts')
from mcrs.retrieval_modules.bge_m3_format import format_query_text, format_track_text
from build_bi_encoder_training_data import _iter_conversation_turns

BGE_REPO = 'OrRim123/recsys2026-bge-m3-music-v1-merged'
CE_REPO = 'OrRim123/recsys2026-bge-reranker-music-v1'
CATALOG_PKL = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local/{BGE_REPO.replace("/","_")}/bge-m3-music-v1-merged/track_embeddings.pkl'
with open(CATALOG_PKL, 'rb') as f:
    payload = pickle.load(f)
track_ids, track_mat = payload['track_ids'], payload['track_mat']
tid_to_idx = {t: i for i, t in enumerate(track_ids)}
tm = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
tid_to_text = {r['track_id']: format_track_text(r.get('track_name','unknown'), r.get('artist_name'), r.get('album_name'), r.get('release_date'), r.get('tag_list')) for r in tm}

bi = SentenceTransformer(BGE_REPO, device='cuda')
ce = CrossEncoder(CE_REPO, device='cuda', max_length=512)

# NOTE: HF splits this dataset as 'train' and 'test'. The 'test' split IS
# the dev set for our purposes — the actual blind evaluation uses
# separate Blind-A / Blind-B datasets. All in-repo code uses split='test'.
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
rows = _iter_conversation_turns(dev)[:500]
queries = [format_query_text(r.get('chat_history') or [], r.get('current_user_query',''), r.get('user_profile_raw'), r.get('conversation_goal'), mode='bge_m3_structured') for r in rows]

# Bi-encoder top-100
q_emb = bi.encode(queries, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype=np.float32)
sims = q_emb @ track_mat.T
top100_idx = np.argpartition(-sims, kth=99, axis=1)[:, :100]
ri = np.arange(sims.shape[0])[:, None]
top100_sorted = top100_idx[ri, np.argsort(-sims[ri, top100_idx], axis=1)]

# Cross-encoder rerank
ndcgs = []
for i, r in enumerate(rows):
    gold = r['track_id']
    cand_tids = [track_ids[j] for j in top100_sorted[i]]
    pairs = [(queries[i], tid_to_text.get(t, '')) for t in cand_tids]
    scores = ce.predict(pairs, batch_size=32, show_progress_bar=False)
    order = np.argsort(-scores)[:20]
    top20 = [cand_tids[j] for j in order]
    if gold in top20:
        rank = top20.index(gold) + 1
        ndcgs.append(1.0 / math.log2(rank + 1))
    else:
        ndcgs.append(0.0)
mean_ndcg = float(sum(ndcgs) / len(ndcgs))
print(f'Stage A+B nDCG@20 on dev: {mean_ndcg:.4f}  (gate: >= 0.25)')
print('PASS' if mean_ndcg >= 0.25 else 'FAIL — investigate before Submission 2.')